# CNN-Based Patient Age Regression from NIH ChestX-ray14

## CourseWork Project

This project aims to develop a Convolutional Neural Network (CNN) for predicting patient age from chest X-ray images.

The project focuses on image regression rather than image classification. The model receives a chest X-ray image as input and predicts one continuous numerical value representing the patient's age.

## 1. Problem Definition

Chest X-ray images contain visual information related to the patient's health condition and biological characteristics.

In this project, we investigate whether a CNN can learn visual patterns from chest X-ray images to estimate patient age.

This is formulated as a regression problem:

- Input: Chest X-ray image
- Target: Patient age
- Output: One continuous numerical value
- Task type: Supervised learning and regression

## 2. Dataset and Target

The project uses the NIH ChestX-ray14 dataset and its corresponding metadata.

The metadata contains patient-level information, including patient age and image identifiers.

### Selected Prediction Target

The selected target is:

> Patient Age

Patient age is selected because it is available in the dataset metadata and can be directly formulated as a continuous regression target.

### Important Data Consideration

The dataset must be split at the patient level rather than randomly splitting individual images. This prevents images belonging to the same patient from appearing in both training and validation/test sets.

## 3. Project Objectives

The main objectives are:

1. Prepare and clean the NIH ChestX-ray14 metadata.
2. Construct a patient-level train/validation/test split.
3. Build a CNN-based regression model.
4. Use Global Average Pooling to reduce spatial feature dimensions.
5. Use a final linear layer to predict patient age.
6. Compare Mean Squared Error and Mean Absolute Error losses.
7. Investigate the effect of image augmentation.
8. Evaluate the model using MAE, MSE, and RMSE.
9. Analyze prediction errors and discuss limitations.

## 4. Proposed CNN Architecture

The proposed model follows the required architecture:

Chest X-ray Image
        ↓
Convolutional Block 1
        ↓
Convolutional Block 2
        ↓
Convolutional Block 3
        ↓
Convolutional Block 4
        ↓
Global Average Pooling
        ↓
Linear Layer
        ↓
Predicted Patient Age

The final output contains one value because the task is regression.

## 5. Project Workflow

The overall workflow is organized into the following stages:

1. Dataset preparation
2. Metadata cleaning
3. Patient-level data splitting
4. Image preprocessing
5. DataLoader construction
6. CNN model implementation
7. Model training
8. Loss function comparison
9. Data augmentation comparison
10. Model evaluation
11. Error analysis
12. Final discussion and conclusion

In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


SEED = 42
set_seed(SEED)

print(f"Random seed is set to {SEED}")

Random seed is set to 42


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Available device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available. The project will run on CPU.")

Available device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [4]:
CONFIG = {
    "seed": 42,
    "image_size": 224,
    "image_channels": 1,
    "batch_size": 32,
    "num_workers": 0,
    "learning_rate": 1e-3,
    "num_epochs": 10,
    "weight_decay": 1e-4,
    "target_column": "Patient Age",
    "task": "regression",
}

CONFIG

{'seed': 42,
 'image_size': 224,
 'image_channels': 1,
 'batch_size': 32,
 'num_workers': 0,
 'learning_rate': 0.001,
 'num_epochs': 10,
 'weight_decay': 0.0001,
 'target_column': 'Patient Age',
 'task': 'regression'}

In [5]:
""" điều chỉnh đường dẫn sao cho phù hợp với repo thực tế."""
PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

METADATA_DIR = RAW_DATA_DIR / "metadata"
IMAGE_DIR = RAW_DATA_DIR / "images"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
FIGURE_DIR = OUTPUT_DIR / "figures"

for folder in [
    DATA_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    OUTPUT_DIR,
    CHECKPOINT_DIR,
    FIGURE_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DATA_DIR)
print("Processed data directory:", PROCESSED_DATA_DIR)
print("Output directory:", OUTPUT_DIR)

Project root: d:\GTVT-UTH\HK_He_2026\DEEP_LEARNING\UTH-Deep-Learning-nhom2\CourseWork\Member_01_Group_Leader\notebook
Raw data directory: d:\GTVT-UTH\HK_He_2026\DEEP_LEARNING\UTH-Deep-Learning-nhom2\CourseWork\Member_01_Group_Leader\notebook\data\raw
Processed data directory: d:\GTVT-UTH\HK_He_2026\DEEP_LEARNING\UTH-Deep-Learning-nhom2\CourseWork\Member_01_Group_Leader\notebook\data\processed
Output directory: d:\GTVT-UTH\HK_He_2026\DEEP_LEARNING\UTH-Deep-Learning-nhom2\CourseWork\Member_01_Group_Leader\notebook\outputs


In [6]:
required_directories = {
    "Raw data": RAW_DATA_DIR,
    "Processed data": PROCESSED_DATA_DIR,
    "Metadata": METADATA_DIR,
    "Images": IMAGE_DIR,
    "Outputs": OUTPUT_DIR,
}

for name, path in required_directories.items():
    print(f"{name}: {path}")
    print("Exists:", path.exists())
    print("-" * 50)

Raw data: d:\GTVT-UTH\HK_He_2026\DEEP_LEARNING\UTH-Deep-Learning-nhom2\CourseWork\Member_01_Group_Leader\notebook\data\raw
Exists: True
--------------------------------------------------
Processed data: d:\GTVT-UTH\HK_He_2026\DEEP_LEARNING\UTH-Deep-Learning-nhom2\CourseWork\Member_01_Group_Leader\notebook\data\processed
Exists: True
--------------------------------------------------
Metadata: d:\GTVT-UTH\HK_He_2026\DEEP_LEARNING\UTH-Deep-Learning-nhom2\CourseWork\Member_01_Group_Leader\notebook\data\raw\metadata
Exists: False
--------------------------------------------------
Images: d:\GTVT-UTH\HK_He_2026\DEEP_LEARNING\UTH-Deep-Learning-nhom2\CourseWork\Member_01_Group_Leader\notebook\data\raw\images
Exists: False
--------------------------------------------------
Outputs: d:\GTVT-UTH\HK_He_2026\DEEP_LEARNING\UTH-Deep-Learning-nhom2\CourseWork\Member_01_Group_Leader\notebook\outputs
Exists: True
--------------------------------------------------


In [7]:
metadata_files = list(METADATA_DIR.glob("*.csv"))

print(f"Number of metadata CSV files found: {len(metadata_files)}")

for file_path in metadata_files:
    print(file_path)

Number of metadata CSV files found: 0


In [8]:
if "metadata_df" in globals():
    print("Columns in metadata:")
    print(metadata_df.columns.tolist())
else:
    print("metadata_df is not available yet.")
"""thành viên 2 xác nhận tên cột, nhóm trưởng có thể kiểm tra:"""

metadata_df is not available yet.


'thành viên 2 xác nhận tên cột, nhóm trưởng có thể kiểm tra:'

In [9]:
def calculate_regression_metrics(y_true, y_pred):
    """
    Calculate MAE, MSE, and RMSE for regression predictions.
    """
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)

    mae = np.mean(np.abs(y_true - y_pred))
    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)

    return {
        "MAE": float(mae),
        "MSE": float(mse),
        "RMSE": float(rmse),
    }

In [10]:
class BaselineCNNRegressor(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
        )

        self.global_average_pooling = nn.AdaptiveAvgPool2d((1, 1))

        self.regressor = nn.Linear(256, 1)

    def forward(self, x):
        x = self.features(x)
        x = self.global_average_pooling(x)
        x = torch.flatten(x, start_dim=1)
        x = self.regressor(x)

        return x

In [11]:
model = BaselineCNNRegressor().to(device)

print(model)

BaselineCNNRegressor(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum

In [ ]:
"""Dữ liệu giả"""
dummy_input = torch.randn(
    2,
    CONFIG["image_channels"],
    CONFIG["image_size"],
    CONFIG["image_size"]
).to(device)

dummy_output = model(dummy_input)

print("Input shape:", dummy_input.shape)
print("Output shape:", dummy_output.shape)

Input shape: torch.Size([2, 1, 224, 224])
Output shape: torch.Size([2, 1])


In [13]:
mse_loss = nn.MSELoss()
mae_loss = nn.L1Loss()

sample_predictions = torch.tensor([[25.0], [40.0], [60.0]])
sample_targets = torch.tensor([[20.0], [45.0], [55.0]])

mse_value = mse_loss(sample_predictions, sample_targets)
mae_value = mae_loss(sample_predictions, sample_targets)

print("MSE loss:", mse_value.item())
print("MAE loss:", mae_value.item())

MSE loss: 25.0
MAE loss: 5.0


## 6. Loss Function Comparison

Two regression loss functions will be investigated.

### Mean Absolute Error

MAE calculates the average absolute difference between the predicted and actual values.

MAE gives a linear penalty to prediction errors and is generally less sensitive to extreme errors.

### Mean Squared Error

MSE calculates the average squared difference between the predicted and actual values.

MSE gives a larger penalty to large errors because the errors are squared.

### Planned Comparison

The project will compare:

1. CNN trained with MSE loss
2. CNN trained with MAE loss

The model architecture, data split, optimizer, learning rate, batch size, and number of epochs should remain unchanged during this comparison.

## 7. Team Member Responsibilities

### Tấn Lên — Group Leader

- Define the project problem and objectives
- Establish the project structure
- Define the common configuration
- Integrate the work of all members
- Review consistency between experiments
- Prepare the final presentation and report

### mạnh Quý — Data Preparation

- Download and organize the dataset
- Clean the metadata
- Extract and validate patient age
- Remove invalid age values
- Create patient-level train/validation/test splits
- Implement the Dataset and DataLoader

###  Lam Linh — CNN Model

- Implement the CNN architecture
- Use convolutional blocks
- Apply Global Average Pooling
- Add the final linear regression layer
- Test the model output shape

###  Ngọc Lan — Loss Function Experiment

- Train the model with MSE loss
- Train the model with MAE loss
- Record training and validation results
- Compare the behavior of both loss functions

### Văn Sơn — Data Augmentation

- Design appropriate image augmentation
- Train models with and without augmentation
- Compare the effect of augmentation
- Analyze whether augmentation improves generalization

###  Thành THi — Evaluation and Analysis

- Calculate MAE, MSE, and RMSE
- Plot training and validation curves
- Create actual-versus-predicted plots
- Analyze prediction errors
- Summarize limitations and conclusions

## 8. Expected Outputs

The final project is expected to produce:

1. A cleaned metadata file
2. Patient-level train/validation/test splits
3. A working CNN regression model
4. Training and validation loss curves
5. MAE, MSE, and RMSE evaluation results
6. A comparison between MSE and MAE
7. A comparison between augmented and non-augmented training
8. Actual-versus-predicted visualization
9. Error analysis
10. Final conclusions and limitations

## 9. Conclusion

This notebook establishes the common foundation for the CNN-based patient age regression project.

The project will use a CNN with Global Average Pooling and a final linear layer to predict patient age from chest X-ray images.

The following members will independently develop the data preparation, model implementation, loss comparison, augmentation experiments, and evaluation components.

All members must follow the shared configuration and patient-level splitting strategy to ensure that the experiments are consistent and reliable.